In [1]:
%%capture
!pip install -q "transformers==4.51.3" "trl==0.18.2" "peft==0.14.0" "bitsandbytes>=0.49.0" "accelerate>=1.0.0" datasets

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- Base model ----
MODEL_NAME      = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH  = 1024

# ---- LoRA ----
LORA_R          = 32
LORA_ALPHA      = 32
LORA_DROPOUT    = 0

# ---- Training ----
BATCH_SIZE      = 1
GRAD_ACCUM      = 16
LEARNING_RATE   = 1e-4
MAX_STEPS       = 500

# ---- Paths ----
DATASET_DIR     = "/kaggle/input/datasets/maximuz23/osint-ai-dataset"
TRAIN_FILE      = f"{DATASET_DIR}/train.jsonl"
VALID_FILE      = f"{DATASET_DIR}/valid.jsonl"
OUTPUT_DIR      = "/kaggle/working/osint-ai"
HF_REPO         = "Maximuz23/Text-OSINT"

# ---- HuggingFace token ----
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map={"": 0},
    token=HF_TOKEN,
    torch_dtype=torch.float16,
)
model.config.use_cache = False

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

2026-06-07 09:31:30.333158: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780824690.521831      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780824690.574333      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780824691.035212      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780824691.035238      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780824691.035241      22 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = "CAUSAL_LM",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

In [5]:
from datasets import load_dataset

raw = load_dataset("json", data_files={
    "train": TRAIN_FILE,
    "valid": VALID_FILE,
})

def format_messages(examples):
    return {"text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in examples["messages"]
    ]}

train_dataset = raw["train"].map(format_messages, batched=True, remove_columns=raw["train"].column_names)
valid_dataset = raw["valid"].map(format_messages, batched=True, remove_columns=raw["valid"].column_names)

print(f"Train set: {len(train_dataset):,}")
print(f"Valid set: {len(valid_dataset):,}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating valid split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/13918 [00:00<?, ? examples/s]

Map:   0%|          | 0/1238 [00:00<?, ? examples/s]

Train set: 13,918
Valid set: 1,238


In [6]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

EVAL_STEPS = 100

sft_config = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    max_steps                   = MAX_STEPS,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = "cosine",
    weight_decay                = 0.01,
    fp16                        = True,
    bf16                        = False,
    optim                       = "paged_adamw_8bit",
    seed                        = 42,
    logging_steps               = 25,
    eval_strategy               = "steps",
    eval_steps                  = EVAL_STEPS,
    save_strategy               = "steps",
    save_steps                  = EVAL_STEPS,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    gradient_checkpointing      = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    dataset_text_field          = "text",
    max_length                  = MAX_SEQ_LENGTH,
    packing                     = True,
    dataset_num_proc            = 1,
)

trainer = SFTTrainer(
    model            = model,
    args             = sft_config,
    train_dataset    = train_dataset,
    eval_dataset     = valid_dataset,
    processing_class = tokenizer,
    callbacks        = [EarlyStoppingCallback(
        early_stopping_patience  = 2,
        early_stopping_threshold = 0.001,
    )],
)

Converting train dataset to ChatML (num_proc=1):   0%|          | 0/13918 [00:00<?, ? examples/s]

Adding EOS to train dataset (num_proc=1):   0%|          | 0/13918 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/13918 [00:00<?, ? examples/s]

Packing train dataset (num_proc=1):   0%|          | 0/13918 [00:00<?, ? examples/s]

Converting eval dataset to ChatML (num_proc=1):   0%|          | 0/1238 [00:00<?, ? examples/s]

Adding EOS to eval dataset (num_proc=1):   0%|          | 0/1238 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=1):   0%|          | 0/1238 [00:00<?, ? examples/s]

Packing eval dataset (num_proc=1):   0%|          | 0/1238 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [7]:
stats = trainer.train()

print(f"\nRuntime {stats.metrics['train_runtime']/3600:.2f}h | final train loss {stats.metrics['train_loss']:.4f}\n")

hist = {}
for e in trainer.state.log_history:
    s = e.get("step")
    if s is None:
        continue
    if "loss" in e:
        hist.setdefault(s, {})["train"] = e["loss"]
    if "eval_loss" in e:
        hist.setdefault(s, {})["valid"] = e["eval_loss"]

print(f"{'step':>6} {'train_loss':>12} {'valid_loss':>12}")
print("-" * 32)
for s in sorted(hist):
    t = hist[s].get("train")
    v = hist[s].get("valid")
    ts = f"{t:.4f}" if t is not None else "-"
    vs = f"{v:.4f}" if v is not None else "-"
    print(f"{s:>6} {ts:>12} {vs:>12}")

best = min((v["valid"], s) for s, v in hist.items() if "valid" in v)
print(f"\nbest valid_loss {best[0]:.4f} at step {best[1]}")


Step,Training Loss,Validation Loss
100,1.626200,1.610463
200,1.602500,1.553427
300,1.556400,1.532885
400,1.545500,1.523084
500,1.547400,1.521166



Runtime 5.84h | final train loss 1.6536

  step   train_loss   valid_loss
--------------------------------
    25       2.7319            -
    50       1.9150            -
    75       1.7391            -
   100       1.6262       1.6105
   125       1.6178            -
   150       1.6463            -
   175       1.5732            -
   200       1.6025       1.5534
   225       1.6516            -
   250       1.5646            -
   275       1.5429            -
   300       1.5564       1.5329
   325       1.5002            -
   350       1.6295            -
   375       1.5886            -
   400       1.5455       1.5231
   425       1.5175            -
   450       1.4636            -
   475       1.5130            -
   500       1.5474       1.5212

best valid_loss 1.5212 at step 500


In [8]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Maximuz23/Text-OSINT/commit/20760ecd9ed2b456355454b82be48f96d345254e', commit_message='Upload tokenizer', commit_description='', oid='20760ecd9ed2b456355454b82be48f96d345254e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Maximuz23/Text-OSINT', endpoint='https://huggingface.co', repo_type='model', repo_id='Maximuz23/Text-OSINT'), pr_revision=None, pr_num=None)